# Batched Sliced Inference Speed with SAHI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obss/sahi/blob/main/demo/inference_with_batch_slicing.ipynb)

A single downscaled pass over a large image misses almost every small object in
it. Slicing finds them, and `batch_size` buys back much of what the extra slices
cost. This notebook measures both, then checks that batching leaves the
detections alone.

The finer you slice, the more you find and the more batching pays, so the two
are measured together across several slice sizes. Every number comes from your
own run.

## Setup

In [ ]:
!pip install -U sahi ultralytics

In [ ]:
import time
from collections.abc import Callable

import cv2
import numpy as np
import torch
from IPython.display import Image as ShowImage
from PIL import Image

from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction
from sahi.prediction import PredictionResult
from sahi.slicing import get_slice_bboxes
from sahi.utils.cv import read_image_as_pil, visualize_object_predictions
from sahi.utils.file import download_from_url

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch:", torch.__version__)

## A large test image

Slicing pays off when the source is much bigger than the detector input, so tile
the demo image into one large scene instead of downloading a big one.

In [ ]:
download_from_url(
    "https://raw.githubusercontent.com/obss/sahi/main/demo/demo_data/small-vehicles1.jpeg",
    "demo_data/small-vehicles1.jpeg",
)

TILES = 3
tile = Image.open("demo_data/small-vehicles1.jpeg").convert("RGB")
large = Image.new("RGB", (tile.width * TILES, tile.height * TILES))
for x in range(TILES):
    for y in range(TILES):
        large.paste(tile, (x * tile.width, y * tile.height))
large.save("demo_data/large.jpg", quality=90)

IMAGE = "demo_data/large.jpg"
WIDTH, HEIGHT = large.size
print("test image:", large.size)

## Helpers

The first call pays for CUDA context setup and autotuning, so warm up once and
keep the best of a few runs rather than an average one stall would dominate.

To compare two runs, detections have to be paired up first. Sorting both lists
and comparing them position by position looks tempting and is wrong: two nearby
objects can swap places and the comparison then measures the distance between
different cars. Pair them by overlap instead.

In [ ]:
def benchmark(fn: Callable, repeat: int = 3) -> tuple:
    """Return (best milliseconds, last result)."""
    result = fn()
    samples = []
    for _ in range(repeat):
        start = time.perf_counter()
        result = fn()
        samples.append(time.perf_counter() - start)
    return min(samples) * 1000, result


def detections(result: PredictionResult) -> np.ndarray:
    """Detections as an (N, 6) array of x1, y1, x2, y2, score, category."""
    rows = [[*p.bbox.to_xyxy(), p.score.value, p.category.id] for p in result.object_prediction_list]
    return np.array(rows, dtype=float)


def iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Pairwise IoU between two sets of boxes."""
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    overlap = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return overlap / (area_a[:, None] + area_b[None, :] - overlap + 1e-12)


def compare(reference: np.ndarray, current: np.ndarray, min_iou: float = 0.9) -> dict:
    """Pair detections by overlap and report how far the paired ones moved."""
    scores = iou_matrix(reference[:, :4], current[:, :4])
    taken, pairs = set(), []
    for i in np.argsort(-scores.max(axis=1)):
        candidates = np.where([j in taken for j in range(scores.shape[1])], -1.0, scores[i])
        j = int(np.argmax(candidates))
        if candidates[j] >= min_iou:
            taken.add(j)
            pairs.append((i, j))
    left = np.array([i for i, _ in pairs], dtype=int)
    right = np.array([j for _, j in pairs], dtype=int)
    return {
        "matched": len(pairs),
        "box_shift": np.abs(reference[left, :4] - current[right, :4]).max() if pairs else 0.0,
        "score_shift": np.abs(reference[left, 4] - current[right, 4]).max() if pairs else 0.0,
        "category_flips": int((reference[left, 5] != current[right, 5]).sum()),
    }

## What slicing buys

One standard pass resizes the whole scene down to the detector input, so a car a
few dozen pixels across arrives as a smudge. Slicing runs the detector over tiles
at native resolution, and a smaller tile means less downscaling still.

In [ ]:
model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolo26n.pt",
    confidence_threshold=0.3,
    device=DEVICE,
)

SLICE_SIZES = [512, 320, 256]
BATCH_SIZES = [1, 4, 8, 16]
OVERLAP = dict(overlap_height_ratio=0.2, overlap_width_ratio=0.2)


def slice_config(size: int) -> dict:
    return dict(slice_height=size, slice_width=size, **OVERLAP)


standard_ms, standard = benchmark(lambda: get_prediction(IMAGE, model))
n_standard = len(standard.object_prediction_list)

# every (slice size, batch size) pair, reused by the sections below
sweep = {}
for size in SLICE_SIZES:
    for batch_size in BATCH_SIZES:
        sweep[size, batch_size] = benchmark(
            lambda s=size, b=batch_size: get_sliced_prediction(IMAGE, model, batch_size=b, verbose=0, **slice_config(s))
        )

print(f"no slicing: {standard_ms:7.1f} ms {n_standard:5d} detections\n")
print(f"{'slice':>6} {'slices':>7} {'detections':>12} {'vs no slicing':>15}")
for size in SLICE_SIZES:
    _, result = sweep[size, BATCH_SIZES[0]]
    found = len(result.object_prediction_list)
    n_slices = len(get_slice_bboxes(HEIGHT, WIDTH, size, size, True, **OVERLAP))
    print(f"{size:6d} {n_slices:7d} {found:12d} {found / max(n_standard, 1):14.1f}x")

## Seeing it

The table says slicing finds more. This is what that looks like on one patch of the
scene, at the three settings side by side.

In [ ]:
BASE = np.ascontiguousarray(read_image_as_pil(IMAGE))
CROP = (430, 130, 1010, 430)  # a stretch of motorway inside the first tile


def count_inside(result: PredictionResult, crop: tuple) -> int:
    """Detections fully inside the crop, so the caption matches what is drawn."""
    x1, y1, x2, y2 = crop
    boxes = detections(result)[:, :4]
    return int(((boxes[:, 0] >= x1) & (boxes[:, 2] <= x2) & (boxes[:, 1] >= y1) & (boxes[:, 3] <= y2)).sum())


def panel(result: PredictionResult, title: str, crop: tuple, zoom: int = 2) -> np.ndarray:
    drawn = visualize_object_predictions(
        BASE.copy(), result.object_prediction_list, rect_th=2, text_size=0.4, hide_conf=True
    )["image"]
    x1, y1, x2, y2 = crop
    view = np.ascontiguousarray(drawn[y1:y2, x1:x2])
    view = cv2.resize(view, (view.shape[1] * zoom, view.shape[0] * zoom), interpolation=cv2.INTER_CUBIC)
    caption = np.full((34, view.shape[1], 3), 255, np.uint8)
    cv2.putText(caption, title, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
    return np.vstack([caption, view])


panels = [panel(standard, f"no slicing: {count_inside(standard, CROP)}", CROP)]
for size in (SLICE_SIZES[0], SLICE_SIZES[-1]):
    result = sweep[size, 4][1]
    panels.append(panel(result, f"slice {size}: {count_inside(result, CROP)}", CROP))

gap = np.full((panels[0].shape[0], 10, 3), 255, np.uint8)
strip = panels[0]
for extra in panels[1:]:
    strip = np.hstack([strip, gap, extra])

# jpeg keeps the stored output small enough to scroll comfortably
cv2.imwrite("demo_data/compare.jpg", cv2.cvtColor(strip, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 90])
ShowImage("demo_data/compare.jpg", width=1000)

Same patch, same model, same confidence threshold. The cars the single pass misses
are not faint or ambiguous, they are simply too small to survive being resized down
to the detector input. Slicing gives them enough pixels to be found.

## Batch size

Each slice is a separate forward pass, and every pass carries fixed overhead:
launching kernels, moving the tile to the GPU, waiting for the result. `batch_size`
decides how many slices go over at once, which spreads that overhead across all of
them instead of paying it per slice.

So the win grows with the number of slices. At a coarse slice size there is little
overhead to amortize, while fine slicing is exactly where batching earns its keep.

In [ ]:
print(f"{'slice':>6} {'slices':>7}  " + "  ".join(f"batch {b:<3d}" for b in BATCH_SIZES) + "   best")
for size in SLICE_SIZES:
    n_slices = len(get_slice_bboxes(HEIGHT, WIDTH, size, size, True, **OVERLAP))
    times = {b: sweep[size, b][0] for b in BATCH_SIZES}
    baseline = times[BATCH_SIZES[0]]
    best = min(times, key=times.get)
    cells = "  ".join(f"{times[b]:6.0f} ms" for b in BATCH_SIZES)
    print(f"{size:6d} {n_slices:7d}  {cells}   {baseline / times[best]:.2f}x at batch {best}")

print("\nTimes are for the whole sliced prediction, so the speedup is what you actually wait for.")

## Batching does not change the result

Speed is only worth having if the detections survive it. Batched and unbatched runs
go through the same slicing and the same postprocessing, so every detection should
come back in the same place with the same label.

Boxes agree to a fraction of a pixel rather than bit for bit, because a convolution
over a batch does not accumulate floating point in the same order as one over a
single image. That is hardware behaviour, and it is orders of magnitude below the
precision of a bounding box.

In [ ]:
for size in SLICE_SIZES:
    reference = detections(sweep[size, BATCH_SIZES[0]][1])
    print(f"\nslice {size}, {len(reference)} detections unbatched")
    print(f"{'batch':>6} {'found':>7} {'matched':>9} {'box shift':>11} {'score shift':>13} {'class flips':>13}")
    for batch_size in BATCH_SIZES:
        current = detections(sweep[size, batch_size][1])
        report = compare(reference, current)
        print(
            f"{batch_size:6d} {len(current):7d} {report['matched']:9d} {report['box_shift']:11.4f} "
            f"{report['score_shift']:13.6f} {report['category_flips']:13d}"
        )

        assert report["matched"] >= 0.99 * len(reference), f"batch {batch_size} lost detections at slice {size}"
        assert report["box_shift"] < 1.0, f"batch {batch_size} moved a box by over a pixel at slice {size}"

No box moves by a pixel at any slice size, and at the coarser two every detection
comes back with its class unchanged.

The finest slice size is where the last columns stop being zero, and neither case is
a batching fault. Both are the model sitting on a boundary that a shift of 0.001 in
the score is enough to cross. A detection scoring within a thousandth of the 0.3
threshold drops below it, which is the missing one. A vehicle the model scores
almost equally as car and as truck swaps between the two, which is a class flip: the
box is identical, only the label changes. Raising the confidence threshold slightly
removes both.

## Where the time goes

A sliced prediction is not one long GPU burn. SAHI reports the split, and it
explains what a faster model can and cannot do for the total.

In [ ]:
PRIMARY = 512
_, primary = sweep[PRIMARY, 8]
parts = {k: v * 1000 for k, v in primary.durations_in_seconds.items()}
total = sum(parts.values())

for name, ms in parts.items():
    print(f"{name:>12}: {ms:7.1f} ms  {ms / total:5.0%}")
print(f"{'total':>12}: {total:7.1f} ms")

## TensorRT

TensorRT compiles the model for one specific GPU, so the engine has to be built on
the machine that runs it. This section skips itself without an NVIDIA GPU and the
`tensorrt` package.

Export with `dynamic=True`. Slicing does not always hand the model a full batch: the
last batch is whatever is left over, and the standard full image pass is a batch of
one. An engine built for a fixed batch rejects those with a shape assertion.

In [ ]:
ENGINE = None
if not torch.cuda.is_available():
    print("skipping TensorRT: no CUDA device")
else:
    try:
        import tensorrt  # noqa: F401
        from ultralytics import YOLO

        ENGINE = YOLO("yolo26n.pt").export(format="engine", imgsz=PRIMARY, batch=8, dynamic=True, device=0)
        print("engine:", ENGINE)
    except ImportError:
        print("skipping TensorRT: not installed, try pip install tensorrt")

In [ ]:
if ENGINE is not None:
    from ultralytics import YOLO

    # the model on its own, away from slicing and postprocessing
    dummy = [np.zeros((PRIMARY, PRIMARY, 3), dtype=np.uint8)] * 8

    def forward(model_path: str) -> float:
        loaded = YOLO(model_path)
        for _ in range(3):
            loaded.predict(dummy, imgsz=PRIMARY, verbose=False, device=0)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(20):
            loaded.predict(dummy, imgsz=PRIMARY, verbose=False, device=0)
        torch.cuda.synchronize()
        return (time.perf_counter() - start) / 20 * 1000

    torch_forward, trt_forward = forward("yolo26n.pt"), forward(str(ENGINE))

    # the same models inside a full sliced prediction
    common = dict(model_type="ultralytics", confidence_threshold=0.3, device=DEVICE, image_size=PRIMARY)
    torch_model = AutoDetectionModel.from_pretrained(model_path="yolo26n.pt", **common)
    trt_model = AutoDetectionModel.from_pretrained(model_path=str(ENGINE), **common)

    def sliced(model: AutoDetectionModel) -> PredictionResult:
        return get_sliced_prediction(IMAGE, model, batch_size=8, verbose=0, **slice_config(PRIMARY))

    torch_ms, torch_result = benchmark(lambda: sliced(torch_model))
    trt_ms, trt_result = benchmark(lambda: sliced(trt_model))

    print(f"{'':>16} {'pytorch':>12} {'tensorrt':>12} {'speedup':>9}")
    print(f"{'forward pass':>16} {torch_forward:9.1f} ms {trt_forward:9.1f} ms {torch_forward / trt_forward:8.2f}x")
    print(f"{'full prediction':>16} {torch_ms:9.1f} ms {trt_ms:9.1f} ms {torch_ms / trt_ms:8.2f}x")
    print(f"\ndetections: {len(torch_result.object_prediction_list)} and {len(trt_result.object_prediction_list)}")
    print("The engine speeds up the model, and the model is only part of what a sliced run does.")

## What to take away

- Slicing is what finds the small objects, many times more than a single pass, and
  a smaller slice finds more still.
- Batching is what makes fine slicing affordable. The gain grows with the number of
  slices, so the setting that costs the most is the one batching helps the most.
- Raise `batch_size` until the timing stops improving. Past that the GPU is already
  saturated and larger batches only cost memory.
- Batching is safe. Every detection comes back, and no box moves by a pixel.
- Time the whole prediction, not just the model. Slicing and postprocessing are real
  work, which is why a faster backend such as TensorRT moves the total by less than
  it moves the forward pass.
- Build TensorRT engines with `dynamic=True`, because slicing produces partial
  batches.